# 01 - Data prep (CWTS)
data shared by dr Kathleen Gregory from CWTS



## Imports

In [ ]:
import re
import requests
import yaml
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from urllib.parse import urlparse
from urllib.parse import unquote
import time


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [ ]:
df_cwts = pd.read_excel('../general/data/banned_data.xlsx', sheet_name='data_us_gov_affils')

In [ ]:
import requests, time
import pandas as pd

MAILTO = "maja.murawka@student.uva.nl"
session = requests.Session()
session.headers.update({"Accept": "application/vnd.api+json","User-Agent": f"metadata-harvest/1.0 (mailto:{MAILTO})"})

unique_dois = (df_cwts["doi"].dropna().astype(str).str.strip().str.lower().unique()).tolist()

rows = []
for doi in unique_dois:
    url = f"https://api.datacite.org/dois/{doi}"
    try:
        r = session.get(url, timeout=30)
        if r.status_code == 429:
            time.sleep(2)
            r = session.get(url, timeout=30)
        if r.status_code != 200:
            rows.append({"doi": doi, "dc_ok": False, "dc_status": r.status_code, "dc_error": r.text[:200]})
            continue

        data = r.json()
        attrs = data.get("data", {}).get("attributes", {})
        rel = data.get("data", {}).get("relationships", {})

        row = {"doi": doi, "dc_ok": True, "dc_status": 200}

        for k in ["prefix","suffix","publisher","publicationYear","language","version","url","contentUrl","schemaVersion","source","isActive","state","reason","metadataVersion","created","registered","published","updated","viewCount","downloadCount","referenceCount","citationCount","partCount","partOfCount","versionCount","versionOfCount"]:
            row[f"dc_{k}"] = attrs.get(k)

        types = attrs.get("types") or {}
        row["dc_types_ris"] = types.get("ris")
        row["dc_types_bibtex"] = types.get("bibtex")
        row["dc_types_citeproc"] = types.get("citeproc")
        row["dc_types_schemaOrg"] = types.get("schemaOrg")
        row["dc_types_resourceTypeGeneral"] = types.get("resourceTypeGeneral")

        row["dc_identifiers"] = attrs.get("identifiers")
        row["dc_alternateIdentifiers"] = attrs.get("alternateIdentifiers")

        titles = attrs.get("titles") or []
        row["dc_title"] = titles[0].get("title") if titles else None
        row["dc_titles_all"] = titles

        creators = attrs.get("creators") or []
        row["dc_creators_all"] = creators
        row["dc_creators_names"] = "; ".join([c.get("name","") for c in creators if c.get("name")]) or None
        row["dc_creators_types"] = "; ".join([c.get("nameType","") for c in creators if c.get("nameType")]) or None

        contributors = attrs.get("contributors") or []
        row["dc_contributors_all"] = contributors
        row["dc_contributors_names"] = "; ".join([c.get("name","") for c in contributors if c.get("name")]) or None

        subjects = attrs.get("subjects") or []
        row["dc_subjects_all"] = subjects
        row["dc_subjects"] = "; ".join([s.get("subject","") for s in subjects if s.get("subject")]) or None

        dates = attrs.get("dates") or []
        row["dc_dates_all"] = dates
        collected = [d.get("date") for d in dates if (d.get("dateType") or "").lower() == "collected"]
        issued = [d.get("date") for d in dates if (d.get("dateType") or "").lower() == "issued"]
        row["dc_date_collected"] = collected[0] if collected else None
        row["dc_date_issued"] = issued[0] if issued else None

        descs = attrs.get("descriptions") or []
        row["dc_descriptions_all"] = descs
        abstract = [d.get("description") for d in descs if (d.get("descriptionType") or "").lower() == "abstract"]
        row["dc_abstract"] = abstract[0] if abstract else None

        geos = attrs.get("geoLocations") or []
        row["dc_geoLocations_all"] = geos
        row["dc_geo_places"] = "; ".join([g.get("geoLocationPlace","") for g in geos if g.get("geoLocationPlace")]) or None

        row["dc_rightsList"] = attrs.get("rightsList")
        row["dc_fundingReferences"] = attrs.get("fundingReferences")

        relids = attrs.get("relatedIdentifiers") or []
        row["dc_relatedIdentifiers_all"] = relids
        row["dc_isVersionOf"] = "; ".join([x.get("relatedIdentifier","") for x in relids if (x.get("relationType") or "") == "IsVersionOf"]) or None

        def rel_id(name):
            d = rel.get(name, {}).get("data")
            if isinstance(d, dict): return d.get("id")
            if isinstance(d, list): return "; ".join([x.get("id","") for x in d if isinstance(x, dict) and x.get("id")]) or None
            return None

        row["dc_rel_client"] = rel_id("client")
        row["dc_rel_provider"] = rel_id("provider")
        row["dc_rel_media"] = rel_id("media")
        row["dc_rel_versionOf"] = rel_id("versionOf")
        row["dc_rel_versions"] = rel_id("versions")
        row["dc_rel_parts"] = rel_id("parts")
        row["dc_rel_partOf"] = rel_id("partOf")
        row["dc_rel_citations"] = rel_id("citations")
        row["dc_rel_references"] = rel_id("references")

        rows.append(row)

    except Exception as e:
        rows.append({"doi": doi, "dc_ok": False, "dc_status": None, "dc_error": str(e)})

datacite_df = pd.DataFrame(rows)

df_cwts["doi_norm"] = df_cwts["doi"].astype(str).str.strip().str.lower()
datacite_df["doi_norm"] = datacite_df["doi"].astype(str).str.strip().str.lower()

df_cwts = df_cwts.merge(datacite_df.drop(columns=["doi"], errors="ignore"), on="doi_norm", how="left")

In [ ]:
#df_cwts.to_csv('00_data/df_cwts.csv')